# Step 3A - Cell Segmentation with Cellpose (tissue-aware)

Notebook counterpart of the SPACEMAP **Segmentation** tab. Both drive
`spacemap.segmentation_cellpose` and `spacemap.cell_spatial_annotation`, so the
CSV produced here is identical to the one the GUI produces with the same settings.

**What changed:** the Annotation tab now runs *before* segmentation. Each ROI image
therefore has a reviewed annotation GeoJSON beside it (`P11_roi_1.ome.tiff` ->
`P11_roi_1_zones.geojson`) that this notebook uses twice:

1. **As a segmentation mask.** Cellpose is restricted to one class of that file
   (`tissue` by default) - the reviewed region with blur, glare and lint already
   removed. Tiles outside it are never sent to the GPU, so a run is both faster
   and free of cells called in artifact regions.
2. **As spatial annotation.** The other classes (Epithelia, Stroma, Tumor, TLS,
   zones, ...) are written onto the extracted cells **in the same pass** as
   feature extraction, so no separate annotation step is needed before batch
   correction.

**Columns added to the features CSV**

| Setting | Column | Meaning |
|---|---|---|
| Binary | `<label>` | 1 when the cell centroid is inside that annotation |
| ROI ID | `<label>_roi` | Which individual polygon contains the cell (largest = 1) |
| ROI size | `<label>_roi_size` | Area of that polygon in square pixels |
| Distance | `<label>_distance` | Signed distance: **negative inside, positive outside** |
| Bed | `<a>_bed`, `<b>_bed` (+`_bed_id`) | Which of two annotations the cell is closer to |
| Edge | `edge_cell` | Cell lies within a buffer of a reference boundary |

**Outputs** (`outputs/segmentation/cellpose/`): `*_cell_features.csv`,
`geojson/*_cell_boundaries.geojson`, optional masks and PNGs.

## 1. Setup

In [ ]:
import glob
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# Import this checkout's sources first: the environment may hold an editable
# install of a different SPACEMAP checkout.
project_root = Path("..").resolve()
if not (project_root / "src" / "spacemap").is_dir():
    project_root = Path(".").resolve()
src_path = str(project_root / "src")
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)

from spacemap.config import setup_logging, log_step
from spacemap.segmentation_cellpose import (
    SegmentationConfig,
    check_segmentation_outputs_exist,
    find_annotation_geojson,
    get_channel_names,
    get_geojson_class_names,
    image_stem,
    segment_image_cellpose,
)
from spacemap.cell_spatial_annotation import (
    EdgeTagConfig,
    annotate_cell_csv,
    default_rules_for_classes,
    list_annotation_classes,
    rules_to_records,
    spatial_columns_for_rules,
)

logger = setup_logging("segmentation")
print("Project root:", project_root)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available (CPU)")

## 2. Configuration

`ANNOTATION_DIR` is the Annotation tab's step-4 output. `MASK_CLASSES` selects which
polygon bounds the segmentation; `tissue` is the reviewed, artifact-free region.

In [ ]:
# ----- Paths -----
INPUT_DIR = "../outputs/rois/"                                   # ROI images from Step 2
OUTPUT_DIR = "../outputs/segmentation/cellpose/"                 # Segmentation outputs
ANNOTATION_DIR = "../outputs/processed_annotations/stage1b_compartments_zones/"
FILE_PATTERN = "*.ome.tiff"

# ----- Channels -----
DAPI_CHANNEL = "DAPI"
SEGMENTATION_MARKERS = [
    "CD31", "E-cadherin", "CD20", "CD4", "Pan-Cytokeratin", "CD68", "HLA-A",
    "CD11c", "CD8", "CD3e", "CD45", "aSMA", "CD163", "FN1", "CD11b", "CD38", "Vimentin",
]

# ----- Tissue mask -----
USE_ANNOTATION_MASK = True     # restrict Cellpose to the reviewed annotation
MASK_CLASSES = ["tissue"]      # class(es) of the annotation used as the mask
# Classes named artefact/artifact/blood_clot/exclude/necrosis are always cut out.

# ----- Spatial annotation of the cell table -----
ANNOTATE_CELLS = True
TISSUE_LABEL = "tissue"        # annotation defining a tissue piece (numbers the _roi columns)
EDGE_TAGGING = EdgeTagConfig(reference="tissue", buffer_px=100.0, column="edge_cell")
# Set SPATIAL_RULES below, after inspecting which classes the files contain.

# ----- Cellpose -----
config = SegmentationConfig()
config.diameter = 0                 # 0 = automatic
config.flow_threshold = 0.8
config.tile_size = 2048             # outer ROI tile used for stitching
config.tile_overlap = 0.1
config.cellpose_internal_tile_size = 224   # GPU memory control (also affects results - leave at 224)
# Speed settings. Benchmarked on an A100-40GB, 2048px tile at 83% tissue:
#   old default (batch 1, cuDNN off, flow on CPU) .............. 12.6 s/tile
#   flow check moved to the GPU alone ........................... 7.7 s/tile  (1.63x)
#   + GPU batch 8 ............................................... 6.4 s/tile  (1.96x)
#   + cuDNN ..................................................... 5.7 s/tile  (2.22x)
# Cell counts across all of those differed by 0.06%, so this is speed without
# a quality trade-off. Batch 16 was no faster than 8.
config.cellpose_batch_size = 8
config.cudnn_mode = "auto"             # "auto" probes cuDNN, falls back if broken
config.flow_threshold_device = "auto"  # "auto" runs the flow-error check on the GPU
config.adaptive_gpu_memory = True      # shrinks batch, then block size, after an OOM
config.min_tissue_fraction = 0.002  # skip outer tiles with almost no tissue
config.tissue_crop_padding = 64
config.cellpose_model_path = "../data/seg_model/CP_20241014_094617"
config.cellpose_model_type = "custom"      # or a pretrained name such as "cyto3"
config.save_geojson = True
config.save_npy = False
config.save_png = False

OVERWRITE_EXISTING = False

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Input      : {INPUT_DIR}")
print(f"Output     : {OUTPUT_DIR}")
print(f"Annotations: {ANNOTATION_DIR}")
print(f"GPU        : {'enabled' if config.use_gpu else 'disabled'}")

## 3. Match images to their annotation files

`find_annotation_geojson` resolves `P11_roi_1.ome.tiff` to `P11_roi_1_zones.geojson`,
falling back to the other Annotation-tab outputs and finally to any GeoJSON whose name
starts with the image stem. This is exactly what the Segmentation tab shows in its
matching table.

In [ ]:
image_files = sorted(glob.glob(os.path.join(INPUT_DIR, FILE_PATTERN)))
print(f"Found {len(image_files)} image(s)\n")

image_info = {}
rows = []
for index, image_path in enumerate(image_files, start=1):
    filename = os.path.basename(image_path)
    annotation_path = find_annotation_geojson(image_path, ANNOTATION_DIR)
    classes = get_geojson_class_names(annotation_path) if annotation_path else []
    existing = check_segmentation_outputs_exist(image_path, OUTPUT_DIR, config)
    image_info[index] = {
        "path": image_path,
        "filename": filename,
        "stem": image_stem(image_path),
        "annotation": annotation_path,
        "classes": classes,
        "processed": existing["all_exist"],
        "size_gb": os.path.getsize(image_path) / (1024 ** 3),
    }
    rows.append({
        "#": index,
        "Image": filename,
        "Size (GB)": round(image_info[index]["size_gb"], 2),
        "Annotation": os.path.basename(annotation_path) if annotation_path else "— none —",
        "Classes": ", ".join(classes),
        "Processed": "yes" if existing["all_exist"] else "no",
    })

if rows:
    display(pd.DataFrame(rows))

if image_files:
    channels = get_channel_names(image_files[0])
    print(f"\nChannels ({len(channels)}): {channels}")
    print(f"DAPI present: {DAPI_CHANNEL in channels}")
    available = [marker for marker in SEGMENTATION_MARKERS if marker in channels]
    print(f"Segmentation markers available: {len(available)}/{len(SEGMENTATION_MARKERS)}")

## 4. Choose the spatial columns

`default_rules_for_classes` gives membership for every class plus ROI id, ROI size and
distance for the tissue annotation. Edit the rules to add distances (for example to
`Tumor`) or a bed competition between two annotations.

In [ ]:
all_classes = []
for info in image_info.values():
    for class_name in info["classes"]:
        if class_name not in all_classes:
            all_classes.append(class_name)
print("Classes across all annotation files:", all_classes or "none found")

SPATIAL_RULES = default_rules_for_classes(all_classes, tissue_label=TISSUE_LABEL)

# --- Customise here. Examples: ---
for rule in SPATIAL_RULES:
    if rule.label.lower() in {"tumor", "tls"}:
        rule.distance = True          # signed distance to tumour / TLS
# Split cells by whether they are closer to normal or malignant tissue:
# for rule in SPATIAL_RULES:
#     if rule.label == "Normal":
#         rule.bed, rule.bed_target, rule.group_by_roi = True, "Malignant", True

display(pd.DataFrame(rules_to_records(SPATIAL_RULES)))
planned = spatial_columns_for_rules(SPATIAL_RULES, EDGE_TAGGING if EDGE_TAGGING else None)
print(f"\n{len(planned)} column(s) will be added:\n  " + ", ".join(planned))

## 5. Run segmentation

Masking and spatial annotation are attached to the per-image config, so both happen
inside `segment_image_cellpose`: the CSV written to disk is already annotated.

In [ ]:
PROCESS_MODE = "batch"          # "single", "batch" or "list"
SINGLE_IMAGE_INDEX = 1
IMAGE_INDICES_TO_PROCESS = [1, 2]

if PROCESS_MODE == "single":
    selected = [image_info[SINGLE_IMAGE_INDEX]] if SINGLE_IMAGE_INDEX in image_info else []
elif PROCESS_MODE == "list":
    selected = [image_info[i] for i in IMAGE_INDICES_TO_PROCESS if i in image_info]
else:
    selected = list(image_info.values())

to_process = [info for info in selected if OVERWRITE_EXISTING or not info["processed"]]
skipped = [info["filename"] for info in selected if info not in to_process]
for name in skipped:
    print(f"Skipping {name} - already processed")
print(f"\nProcessing {len(to_process)} image(s)")

processing_results = {}
successful = failed = total_cells = 0

for position, info in enumerate(to_process, start=1):
    print(f"\n--- {position}/{len(to_process)}: {info['filename']} ({info['size_gb']:.2f} GB) ---")

    # Mask: restrict Cellpose to the chosen annotation class.
    if USE_ANNOTATION_MASK and info["annotation"]:
        config.use_input_geojson = True
        config.input_geojson_path = info["annotation"]
        config.mask_class_names = list(MASK_CLASSES)
        print(f"  Mask: '{', '.join(MASK_CLASSES)}' from {os.path.basename(info['annotation'])}")
    else:
        config.use_input_geojson = False
        config.mask_class_names = []
        if USE_ANNOTATION_MASK:
            print("  Mask: no annotation matched, segmenting the whole image")

    # Spatial annotation: tag the cells in the same pass.
    if ANNOTATE_CELLS and info["annotation"]:
        available = {name.lower() for name in info["classes"]}
        config.annotation_geojson_path = info["annotation"]
        config.spatial_rules = [
            rule for rule in SPATIAL_RULES if rule.label.lower() in available
        ]
        config.spatial_tissue_label = TISSUE_LABEL
        config.spatial_edge_config = (
            {"reference": EDGE_TAGGING.reference, "buffer": EDGE_TAGGING.buffer_px,
             "column": EDGE_TAGGING.column} if EDGE_TAGGING else None
        )
        print(f"  Spatial annotation: {len(config.spatial_rules)} rule(s)")
    else:
        config.annotation_geojson_path = None
        config.spatial_rules = []
        config.spatial_edge_config = None

    try:
        with log_step(f"Segmenting {info['filename']}", logger):
            results = segment_image_cellpose(
                ome_tiff_path=info["path"],
                output_dir=OUTPUT_DIR,
                config=config,
                markers=SEGMENTATION_MARKERS,
                dapi_channel=DAPI_CHANNEL,
                overwrite=OVERWRITE_EXISTING,
            )
        processing_results[info["filename"]] = results
        successful += 1

        features = results.get("features_df")
        if features is not None:
            total_cells += len(features)
            print(f"  SUCCESS: {len(features):,} cells, mean area {features['area'].mean():.1f} px")
            summary = results.get("spatial_summary") or {}
            if summary.get("counts"):
                inside = ", ".join(f"{k}={v:,}" for k, v in summary["counts"].items())
                print(f"  Cells per annotation: {inside}")
            for warning in summary.get("warnings", []):
                print(f"  Warning: {warning}")
    except Exception as error:
        print(f"  FAILED: {error}")
        processing_results[info["filename"]] = {"error": str(error)}
        failed += 1
        import traceback
        traceback.print_exc()

print("\n" + "=" * 60)
print(f"Succeeded: {successful}/{len(to_process)} | Failed: {failed} | Total cells: {total_cells:,}")
print(f"Outputs in: {OUTPUT_DIR}")

## 6. Inspect the annotated cell table

The features CSV now carries the morphology, the per-marker intensities **and** the
spatial columns, which is exactly the shape Step 5 (batch correction) expects.

In [ ]:
last_csv = None
for filename, result in processing_results.items():
    if isinstance(result, dict) and result.get("features_csv"):
        last_csv = result["features_csv"]

if last_csv and os.path.exists(last_csv):
    df = pd.read_csv(last_csv)
    print(f"{os.path.basename(last_csv)}: {len(df):,} cells x {len(df.columns)} columns\n")

    morphology = ["label", "area", "major_axis_length", "minor_axis_length",
                  "eccentricity", "solidity", "extent", "orientation", "x_centroid", "y_centroid"]
    spatial = [c for c in df.columns if c in set(planned)]
    markers = [c for c in df.columns if c not in morphology + spatial + ["original_label"]]
    print(f"Morphology ({len(morphology)}): {morphology}")
    print(f"Markers    ({len(markers)}): {markers[:10]}{' ...' if len(markers) > 10 else ''}")
    print(f"Spatial    ({len(spatial)}): {spatial}")

    if spatial:
        print("\nCells per annotation:")
        for column in spatial:
            values = df[column]
            if column.endswith("_distance"):
                print(f"  {column:24s} {values.min():8.1f} .. {values.max():8.1f} px "
                      f"({int((values < 0).sum()):,} inside)")
            elif column.endswith(("_roi", "_roi_size", "_bed_id")):
                print(f"  {column:24s} {values.nunique():8,} distinct value(s)")
            elif set(values.dropna().unique()) <= {0, 1}:
                print(f"  {column:24s} {int(values.sum()):8,} cells")
        display(df[["label", "x_centroid", "y_centroid"] + spatial].head(10))
else:
    print("No features CSV available. Run the segmentation cell first.")

## 7. Segmentation quality checks

In [ ]:
if last_csv and os.path.exists(last_csv):
    df = pd.read_csv(last_csv)
    figure, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

    axes[0, 0].hist(df["area"], bins=60, alpha=0.8, edgecolor="black")
    axes[0, 0].set_xlabel("Cell area (px)"); axes[0, 0].set_ylabel("Cells")
    axes[0, 0].set_title("Cell area distribution")

    axes[0, 1].hist(df["eccentricity"], bins=60, alpha=0.8, edgecolor="black", color="orange")
    axes[0, 1].set_xlabel("Eccentricity"); axes[0, 1].set_ylabel("Cells")
    axes[0, 1].set_title("Cell eccentricity distribution")

    # Cells in space, coloured by the tissue domain they were tagged with.
    domain_columns = [
        column for column in df.columns
        if column in set(planned) and not column.endswith(("_roi", "_roi_size", "_distance", "_bed_id"))
        and set(df[column].dropna().unique()) <= {0, 1}
    ]
    axes[1, 0].scatter(df["x_centroid"], df["y_centroid"], s=2, c="#cccccc", label="unassigned")
    for column in domain_columns[:6]:
        subset = df[df[column] == 1]
        if len(subset):
            axes[1, 0].scatter(subset["x_centroid"], subset["y_centroid"], s=2, label=column)
    axes[1, 0].invert_yaxis()
    axes[1, 0].set_aspect("equal")
    axes[1, 0].set_title("Cells coloured by tissue domain")
    axes[1, 0].legend(markerscale=4, fontsize=8, loc="upper right")

    distance_columns = [c for c in df.columns if c.endswith("_distance")]
    if distance_columns:
        column = distance_columns[0]
        axes[1, 1].hist(df[column].dropna(), bins=60, alpha=0.8, edgecolor="black", color="seagreen")
        axes[1, 1].axvline(0, color="crimson", linestyle="--", linewidth=2, label="annotation boundary")
        axes[1, 1].set_xlabel(f"{column} (px; negative = inside)")
        axes[1, 1].set_ylabel("Cells")
        axes[1, 1].set_title(f"Signed distance: {column}")
        axes[1, 1].legend(fontsize=8)
    else:
        axes[1, 1].axis("off")

    plt.show()

    print(f"Cells            : {len(df):,}")
    print(f"Mean area        : {df['area'].mean():.1f} +/- {df['area'].std():.1f} px")
    print(f"Mean eccentricity: {df['eccentricity'].mean():.3f}")
else:
    print("No features CSV available.")

## 8. Annotating an existing CSV

If a cell table was segmented before the annotations existed, or the annotations were
refined in QuPath afterwards, re-tag the CSV in place instead of re-segmenting.
`annotate_cell_csv` uses the same code path as the inline annotation above.

In [ ]:
REANNOTATE = False   # set True to run

if REANNOTATE and image_info:
    for info in image_info.values():
        csv_path = os.path.join(OUTPUT_DIR, f"{info['stem']}.ome_cell_features.csv")
        if not os.path.exists(csv_path):
            csv_path = os.path.join(OUTPUT_DIR, f"{info['stem']}_cell_features.csv")
        if not os.path.exists(csv_path) or not info["annotation"]:
            continue
        available = {name.lower() for name in info["classes"]}
        summary = annotate_cell_csv(
            csv_path=csv_path,
            geojson_path=info["annotation"],       # re-read from disk, so QuPath edits count
            output_path=csv_path,                  # overwrite in place
            rules=[r for r in SPATIAL_RULES if r.label.lower() in available],
            tissue_label=TISSUE_LABEL,
            edge_config=EDGE_TAGGING,
        )
        print(f"{os.path.basename(csv_path)}: {summary['cells']:,} cells, "
              f"{len(summary['columns'])} spatial column(s)")
        for warning in summary.get("warnings", []):
            print(f"  Warning: {warning}")
else:
    print("Set REANNOTATE = True to re-tag existing CSVs.")

## Summary

| Output | Contents |
|---|---|
| `*_cell_features.csv` | morphology + per-marker intensities + spatial annotation columns |
| `geojson/*_cell_boundaries.geojson` | cell outlines for QuPath |
| `*_mask.npy`, `*_segmentation.png` | optional raster outputs |

**Next:** `Step5_batch_correction.ipynb`. The features CSV is already in the expected
shape - the `tissue` column is the usual tissue filter, and the domain columns carry
through into phenotyping and spatial analysis.

**Related notebooks**

* `Step4_annotation.ipynb` - produces the tissue annotations this notebook consumes.
* `Step4B_cell_annotation.ipynb` - the standalone cell-annotation workflow, for cell
  tables that were not annotated during segmentation.